# A1.3 · Authorization models that make bad grants impossible

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

---

**Risk.** RBAC has no unit small enough to express the grant you actually meant.

**Control.** ReBAC/ABAC with time-scoped delegation and attenuation by construction.

**This lab.** Make the over-privileged grant structurally unrepresentable.

| | |
|---|---|
| Open-source tooling | OpenFGA, OPA |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A1.3"))

Most authorization models can express a bad grant. The question for an architect is whether yours can be *made* not to.

In [ ]:
from cybercommons import identity

print("ceilings — what each actor may hold at most, whoever asks:")
for actor, scopes in identity.GRANTS.items():
    print(f"  {actor:16s} {sorted(scopes)}")

alice = identity.mint("alice")
print("\nalice holds:", sorted(alice.scopes))

# the grant a ticket would ask for, and the system refuses to express
for actor, want in [("reviewer-agent", {"repo:write"}),
                    ("reviewer-agent", {"secrets:read"}),
                    ("patch-agent",    {"repo:write"})]:
    try:
        t = identity.exchange(alice, actor, want)
        print(f"  GRANTED  {actor} ← {sorted(want)}")
    except identity.DelegationError as e:
        print(f"  REFUSED  {actor} ← {sorted(want)}: {e}")

The refusal does not depend on anyone reviewing the request. The ceiling makes the bad grant *unrepresentable*, which is the only kind of control that survives a busy quarter.

### Expect

`patch-agent` receives `repo:write`. Both requests for `reviewer-agent` are refused, and the error names the ceiling that refused them — not a policy document, the token exchange itself.

### Your turn

Add a `break-glass` actor with every scope. Now decide what makes it safe: a ceiling cannot, so the control has to be time and audit. Model it with `identity.JITGrant`.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A1.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*